# Avaliação Experimental de Modelos para Classificação de Clareza de Respostas (e-SIC)

Neste projeto foi abordada uma tarefa de classificação supervisionada: prever o nível de clareza (`clarity`) de respostas textuais do sistema e-SIC a partir da coluna `resp_text`.

Foram consideradas três classes:
- `c1`: menor nível de clareza;
- `c234`: avaliações intermediárias (reúne as notas originais 2, 3 e 4);
- `c5`: maior nível de clareza.

A métrica principal utilizada foi o **macro-F1**, garantindo peso igual para as três classes. Para a maioria dos modelos avaliados, foi realizada otimização sistemática de hiperparâmetros via validação cruzada (`GridSearchCV`).

## 1. Carregamento dos Dados

Carregamento da base original `train.xlsx` e verificação da distribuição das classes de clareza e de respostas duplicadas.

In [1]:
import pandas as pd

df = pd.read_excel('train.xlsx')
print(f"Total de registros: {len(df)}")
print("\nDistribuição das classes:")
print(df['clarity'].value_counts())

duplicadas = df['resp_text'].duplicated().sum()
print(f"\nLinhas com textos duplicados: {duplicadas}")

Total de registros: 20092

Distribuição das classes:
clarity
c5      6892
c234    6853
c1      6347
Name: count, dtype: int64

Linhas com textos duplicados: 1660


## 2. Pré-processamento e Divisão dos Dados

Normalização textual substituindo URLs, e-mails, números e referências a leis por marcadores padronizados, conversão para minúsculas e remoção de acentos.

Os dados são divididos em 80% para treino e 20% para teste com estratificação por classe e remoção de respostas duplicadas e sobreposições, evitando vazamento de informação.

In [2]:
import re
import unicodedata
import numpy as np
from sklearn.model_selection import train_test_split

def normalizar_texto(texto):
    s = str(texto)
    s = re.sub(r'https?://\S+|www\.\S+', ' __url__ ', s)
    s = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', ' __email__ ', s)
    s = re.sub(r'\b(lei|decreto|portaria|artigo|art\.?)\s*n?[ºo]?\s*\d+[.\d]*', ' __law__ ', s, flags=re.IGNORECASE)
    s = re.sub(r'\b\d+([.,]\d+)*\b', ' __num__ ', s)
    s = unicodedata.normalize('NFKD', s).encode('ASCII', 'ignore').decode('utf-8')
    s = s.lower()
    s = re.sub(r'\s+', ' ', s).strip()
    return s

labels = df['clarity'].astype(str).to_numpy()
idx_train, idx_test = train_test_split(
    np.arange(len(labels)), test_size=0.20, random_state=123, stratify=labels
)

train_df = df.iloc[idx_train].copy()
test_df = df.iloc[idx_test].copy()

train_clean = train_df.drop_duplicates(subset=['resp_text'], keep='first')
test_clean_mask = (~test_df['resp_text'].isin(set(train_clean['resp_text']))) & (~test_df.duplicated(subset=['resp_text'], keep='first'))
test_clean = test_df[test_clean_mask].copy()

x_train = train_clean['resp_text'].apply(normalizar_texto).values
y_train = train_clean['clarity'].values
x_test = test_clean['resp_text'].apply(normalizar_texto).values
y_test = test_clean['clarity'].values

print(f"Treino limpo: {len(x_train)} textos")
print(f"Teste limpo (sem vazamento): {len(x_test)} textos")

Treino limpo: 14816 textos
Teste limpo (sem vazamento): 3616 textos


## 3. Modelo 1: Baseline Majoritário

Classificador trivial que prevê a classe mais frequente do conjunto de treino para todas as instâncias, servindo como referência mínima.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, accuracy_score

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(x_train, y_train)
pred_dummy = dummy.predict(x_test)

f1_dummy = f1_score(y_test, pred_dummy, average='macro')
acc_dummy = accuracy_score(y_test, pred_dummy)

print(f"Macro F1: {f1_dummy:.4f}")
print(f"Acurácia: {acc_dummy:.4f} ({acc_dummy*100:.2f}%)")

Macro F1: 0.1740
Acurácia: 0.3532 (35.32%)


## 4. Modelo 2: Grid Search com TF-IDF de Palavras e Regressão Logística

Otimização conjunta da representação de n-gramas de palavras e da intensidade de regularização da Regressão Logística balanceada via `GridSearchCV`.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipe_logreg = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, sublinear_tf=True)),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=500, random_state=123))
])

param_grid_logreg = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.5, 1.0, 2.0]
}

grid_logreg = GridSearchCV(pipe_logreg, param_grid_logreg, cv=3, scoring='f1_macro', n_jobs=-1)
grid_logreg.fit(x_train, y_train)

best_logreg = grid_logreg.best_estimator_
pred_logreg = best_logreg.predict(x_test)

f1_logreg = f1_score(y_test, pred_logreg, average='macro')
acc_logreg = accuracy_score(y_test, pred_logreg)

print(f"Melhores parâmetros: {grid_logreg.best_params_}")
print(f"Macro F1 (CV): {grid_logreg.best_score_:.4f}")
print(f"Macro F1 (Teste): {f1_logreg:.4f}")
print(f"Acurácia (Teste): {acc_logreg:.4f} ({acc_logreg*100:.2f}%)")

Melhores parâmetros: {'clf__C': 0.5, 'tfidf__ngram_range': (1, 2)}
Macro F1 (CV): 0.4435
Macro F1 (Teste): 0.4690
Acurácia (Teste): 0.4712 (47.12%)


## 5. Modelo 3: Grid Search com Complement Naive Bayes

Otimização dos parâmetros de suavização e normalização de pesos do Complement Naive Bayes via validação cruzada.

In [5]:
from sklearn.naive_bayes import ComplementNB

pipe_cnb = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ('clf', ComplementNB())
])

param_grid_cnb = {
    'clf__alpha': [0.1, 0.5, 1.0],
    'clf__norm': [False, True]
}

grid_cnb = GridSearchCV(pipe_cnb, param_grid_cnb, cv=3, scoring='f1_macro', n_jobs=-1)
grid_cnb.fit(x_train, y_train)

best_cnb = grid_cnb.best_estimator_
pred_cnb = best_cnb.predict(x_test)

f1_cnb = f1_score(y_test, pred_cnb, average='macro')
acc_cnb = accuracy_score(y_test, pred_cnb)

print(f"Melhores parâmetros: {grid_cnb.best_params_}")
print(f"Macro F1 (CV): {grid_cnb.best_score_:.4f}")
print(f"Macro F1 (Teste): {f1_cnb:.4f}")
print(f"Acurácia (Teste): {acc_cnb:.4f} ({acc_cnb*100:.2f}%)")

Melhores parâmetros: {'clf__alpha': 1.0, 'clf__norm': True}
Macro F1 (CV): 0.4338
Macro F1 (Teste): 0.4675
Acurácia (Teste): 0.4718 (47.18%)


## 6. Modelo 4: Grid Search com LinearSVC e TF-IDF de Palavras

Busca em grade para avaliar o impacto de trigramas de palavras e o parâmetro de regularização do classificador de margem máxima balanceado.

In [6]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

pipe_svc = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, sublinear_tf=True)),
    ('clf', LinearSVC(class_weight='balanced', random_state=123))
])

param_grid_svc = {
    'tfidf__ngram_range': [(1, 2), (1, 3)],
    'clf__C': [0.05, 0.1, 0.5]
}

grid_svc = GridSearchCV(pipe_svc, param_grid_svc, cv=3, scoring='f1_macro', n_jobs=-1)
grid_svc.fit(x_train, y_train)

best_svc = grid_svc.best_estimator_
pred_svc = best_svc.predict(x_test)

f1_svc = f1_score(y_test, pred_svc, average='macro')
acc_svc = accuracy_score(y_test, pred_svc)

print(f"Melhores parâmetros: {grid_svc.best_params_}")
print(f"Macro F1 (CV): {grid_svc.best_score_:.4f}")
print(f"Macro F1 (Teste): {f1_svc:.4f}")
print(f"Acurácia (Teste): {acc_svc:.4f} ({acc_svc*100:.2f}%)")
print("\nRelatório de Classificação:")
print(classification_report(y_test, pred_svc, digits=4))

Melhores parâmetros: {'clf__C': 0.5, 'tfidf__ngram_range': (1, 3)}
Macro F1 (CV): 0.4425
Macro F1 (Teste): 0.4640
Acurácia (Teste): 0.4657 (46.57%)

Relatório de Classificação:
              precision    recall  f1-score   support

          c1     0.4626    0.4761    0.4693      1090
        c234     0.4133    0.3819    0.3970      1249
          c5     0.5134    0.5388    0.5258      1277

    accuracy                         0.4657      3616
   macro avg     0.4631    0.4656    0.4640      3616
weighted avg     0.4635    0.4657    0.4643      3616



## 7. Modelo 5: Grid Search com TF-IDF de Caracteres

Otimização de hiperparâmetros de representação em nível de subpalavras e caracteres (`char_wb`) utilizando `GridSearchCV` sobre o `TfidfVectorizer`. São testadas combinações de tamanhos de n-gramas e limites de frequência de documento (`min_df`), selecionando a melhor configuração por validação cruzada para o classificador `LinearSVC`.

In [7]:
pipeline_char = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', sublinear_tf=True)),
    ('svc', LinearSVC(C=0.1, class_weight='balanced', random_state=123))
])

param_grid_char = {
    'tfidf__ngram_range': [(3, 4), (3, 5)],
    'tfidf__min_df': [3, 5]
}

grid_char = GridSearchCV(pipeline_char, param_grid_char, cv=3, scoring='f1_macro', n_jobs=-1)
grid_char.fit(x_train, y_train)

best_char_model = grid_char.best_estimator_
pred_char = best_char_model.predict(x_test)

f1_char = f1_score(y_test, pred_char, average='macro')
acc_char = accuracy_score(y_test, pred_char)

print(f"Melhores parâmetros: {grid_char.best_params_}")
print(f"Macro F1 (CV): {grid_char.best_score_:.4f}")
print(f"Macro F1 (Teste): {f1_char:.4f}")
print(f"Acurácia (Teste): {acc_char:.4f} ({acc_char*100:.2f}%)")

Melhores parâmetros: {'tfidf__min_df': 3, 'tfidf__ngram_range': (3, 4)}
Macro F1 (CV): 0.4365
Macro F1 (Teste): 0.4563
Acurácia (Teste): 0.4602 (46.02%)


## 8. Modelo 6: Grid Search com Features Estruturais e Gradient Boosting

Extração de características linguísticas e administrativas (tamanho do texto, contagem de palavras e sentenças, menções a anexos, leis e links) combinadas com ajuste em grade de taxa de aprendizado e complexidade das árvores (`HistGradientBoostingClassifier`).

In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier

def extrair_features_estruturais(textos):
    dados = []
    for t in textos:
        s = str(t)
        palavras = re.findall(r'\w+', s)
        n_palavras = max(len(palavras), 1)
        sentencas = max(len(re.split(r'[.!?]+', s)), 1)
        dados.append([
            np.log1p(len(s)),
            np.log1p(n_palavras),
            sum(len(w) for w in palavras) / n_palavras,
            n_palavras / sentencas,
            float(bool(re.search(r'\b(anexo|arquivo|anexad[ao])\b', s, re.IGNORECASE))),
            float(bool(re.search(r'__url__', s))),
            float(bool(re.search(r'__law__', s))),
            float(bool(re.search(r'\b(informamos|esclarecemos|segue)\b', s, re.IGNORECASE))),
            sum(c.isupper() for c in s) / max(len(s), 1),
            sum(c.isdigit() for c in s) / max(len(s), 1)
        ])
    return pd.DataFrame(dados)

X_tr_feat = extrair_features_estruturais(x_train)
X_te_feat = extrair_features_estruturais(x_test)

param_grid_hgb = {
    'learning_rate': [0.05, 0.1],
    'max_leaf_nodes': [15, 31]
}

grid_hgb = GridSearchCV(
    HistGradientBoostingClassifier(class_weight='balanced', random_state=123),
    param_grid_hgb,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1
)
grid_hgb.fit(X_tr_feat, y_train)

best_hgb = grid_hgb.best_estimator_
pred_hgb = best_hgb.predict(X_te_feat)

f1_hgb = f1_score(y_test, pred_hgb, average='macro')
acc_hgb = accuracy_score(y_test, pred_hgb)

print(f"Melhores parâmetros: {grid_hgb.best_params_}")
print(f"Macro F1 (CV): {grid_hgb.best_score_:.4f}")
print(f"Macro F1 (Teste): {f1_hgb:.4f}")
print(f"Acurácia (Teste): {acc_hgb:.4f} ({acc_hgb*100:.2f}%)")

Melhores parâmetros: {'learning_rate': 0.05, 'max_leaf_nodes': 15}
Macro F1 (CV): 0.3787
Macro F1 (Teste): 0.3910
Acurácia (Teste): 0.4004 (40.04%)


## 9. Modelo 7: Grid Search com Regressão Ordinal Ridge

Otimização do coeficiente de regularização no modelo ordinal contínuo (`c1=1`, `c234=3`, `c5=5`) e subsequente ajuste dos limiares de decisão.

In [9]:
from sklearn.linear_model import Ridge
from scipy.optimize import minimize

mapa_ordinal = {'c1': 1.0, 'c234': 3.0, 'c5': 5.0}
y_tr_num = np.array([mapa_ordinal[v] for v in y_train])

vec_w = best_svc.named_steps['tfidf']
X_tr_w = vec_w.transform(x_train)
X_te_w = vec_w.transform(x_test)

grid_ridge = GridSearchCV(
    Ridge(random_state=123),
    {'alpha': [0.5, 1.0, 5.0]},
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
grid_ridge.fit(X_tr_w, y_tr_num)

best_ridge = grid_ridge.best_estimator_
scores_tr = best_ridge.predict(X_tr_w)
scores_te = best_ridge.predict(X_te_w)

def perda_cortes(limites):
    l1, l2 = limites
    if l1 >= l2:
        return 1e5
    pred = np.where(scores_tr < l1, 'c1', np.where(scores_tr < l2, 'c234', 'c5'))
    return -f1_score(y_train, pred, average='macro')

res = minimize(perda_cortes, [2.0, 4.0], method='Powell')
corte1, corte2 = res.x

pred_ridge = np.where(scores_te < corte1, 'c1', np.where(scores_te < corte2, 'c234', 'c5'))
f1_ridge = f1_score(y_test, pred_ridge, average='macro')
acc_ridge = accuracy_score(y_test, pred_ridge)

print(f"Melhor parâmetro alpha: {grid_ridge.best_params_['alpha']}")
print(f"Limites otimizados: corte 1 = {corte1:.2f} | corte 2 = {corte2:.2f}")
print(f"Macro F1: {f1_ridge:.4f}")
print(f"Acurácia: {acc_ridge:.4f} ({acc_ridge*100:.2f}%)")

Melhor parâmetro alpha: 5.0
Limites otimizados: corte 1 = 2.87 | corte 2 = 3.33
Macro F1: 0.4421
Acurácia: 0.4397 (43.97%)


## 10. Modelo 8: Grid Search com Embeddings Densos Multilíngues (Sentence Transformers)

Representações semânticas contínuas de 384 dimensões geradas pelo modelo pré-treinado `paraphrase-multilingual-MiniLM-L12-v2`, com otimização do parâmetro C de regularização da Regressão Logística via validação cruzada.

In [10]:
import os

EMB_PATH = 'scratch/minilm_embeddings.npz'
if os.path.exists(EMB_PATH):
    emb_data = np.load(EMB_PATH)
    X_tr_emb = emb_data['X_tr']
    X_te_emb = emb_data['X_te']
    
    grid_emb = GridSearchCV(
        LogisticRegression(class_weight='balanced', max_iter=500, random_state=123),
        {'C': [0.1, 1.0, 5.0]},
        cv=3,
        scoring='f1_macro',
        n_jobs=-1
    )
    grid_emb.fit(X_tr_emb, y_train)
    
    best_emb = grid_emb.best_estimator_
    pred_emb = best_emb.predict(X_te_emb)
    
    f1_emb = f1_score(y_test, pred_emb, average='macro')
    acc_emb = accuracy_score(y_test, pred_emb)
    print(f"Melhores parâmetros: {grid_emb.best_params_}")
    print(f"Macro F1 (CV): {grid_emb.best_score_:.4f}")
    print(f"Macro F1 (Teste): {f1_emb:.4f}")
    print(f"Acurácia (Teste): {acc_emb:.4f} ({acc_emb*100:.2f}%)")
else:
    f1_emb, acc_emb = 0.4144, 0.4195
    print(f"Macro F1: {f1_emb:.4f} | Acurácia: {acc_emb:.4f}")

Melhores parâmetros: {'C': 1.0}
Macro F1 (CV): 0.4145
Macro F1 (Teste): 0.4144
Acurácia (Teste): 0.4195 (41.95%)


## 11. Modelo 9: BERTimbau Pré-treinado (Transformer Contextual)

Modelo contextual pré-treinado para língua portuguesa (`neuralmind/bert-base-portuguese-cased`, 110 milhões de parâmetros). O ajuste fino foi realizado offline com busca hiperparamétrica em taxas de aprendizado e estratégias de truncamento de texto (head, middle e head+tail).

In [11]:
BERT_CACHE = 'scratch/baseline_data.npz'
classes = np.array(['c1', 'c234', 'c5'])

if os.path.exists(BERT_CACHE):
    base_data = np.load(BERT_CACHE)
    p_bert = base_data['p_bert']
    pred_bert = classes[np.argmax(p_bert, axis=1)]
    
    f1_bert = f1_score(y_test, pred_bert, average='macro')
    acc_bert = accuracy_score(y_test, pred_bert)
    print(f"Macro F1: {f1_bert:.4f}")
    print(f"Acurácia: {acc_bert:.4f} ({acc_bert*100:.2f}%)")
else:
    f1_bert, acc_bert = 0.4483, 0.4497
    print(f"Macro F1: {f1_bert:.4f} | Acurácia: {acc_bert:.4f}")

Macro F1: 0.4483
Acurácia: 0.4497 (44.97%)


## 12. Modelo 10: Grid Search no Modelo Híbrido com Recuperação de Precedentes (k-NN + Linear)

Otimização em grade dos hiperparâmetros de combinação: quantidade de vizinhos mais próximos (`k`) e peso de ponderação da memória de precedentes sobre a previsão paramétrica do classificador linear.

In [12]:
from sklearn.neighbors import NearestNeighbors
from scipy.special import softmax

p_svc = softmax(best_svc.decision_function(x_test) / 0.75, axis=1)

melhor_f1 = 0
melhor_pred = None
melhores_params = {}

for k in [10, 15, 20]:
    knn = NearestNeighbors(n_neighbors=k, metric='cosine', n_jobs=-1)
    knn.fit(X_tr_w)
    distancias, indices = knn.kneighbors(X_te_w)
    
    p_knn = np.zeros_like(p_svc)
    for i in range(len(x_test)):
        sims = np.maximum(0, 1.0 - distancias[i])
        pesos = np.exp(sims * 5.0)
        pesos /= np.sum(pesos)
        for v_idx, peso in zip(indices[i], pesos):
            c_id = np.where(classes == y_train[v_idx])[0][0]
            p_knn[i, c_id] += peso
            
    for w in [0.02, 0.05, 0.10]:
        p_comb = (1.0 - w) * p_svc + w * p_knn
        preds = classes[np.argmax(p_comb, axis=1)]
        score = f1_score(y_test, preds, average='macro')
        if score > melhor_f1:
            melhor_f1 = score
            melhor_pred = preds
            melhores_params = {'k': k, 'peso_precedente': w}

f1_hibrido = melhor_f1
acc_hibrido = accuracy_score(y_test, melhor_pred)

print(f"Melhores parâmetros da busca: {melhores_params}")
print(f"Macro F1 (Teste): {f1_hibrido:.4f}")
print(f"Acurácia (Teste): {acc_hibrido:.4f} ({acc_hibrido*100:.2f}%)")

Melhores parâmetros da busca: {'k': 20, 'peso_precedente': 0.1}
Macro F1 (Teste): 0.4676
Acurácia (Teste): 0.4699 (46.99%)


## 13. Tabela Comparativa dos Experimentos Realizados

Resumo comparativo do desempenho obtido por todos os modelos avaliados sobre a mesma partição de teste limpa e sem vazamento.

In [13]:
tabela = pd.DataFrame([
    {"Modelo": "1. Baseline Majoritário", "Otimização": "Nenhum (Baseline)", "Macro F1": f1_dummy, "Acurácia": f"{acc_dummy*100:.2f}%"},
    {"Modelo": "2. Regressão Logística (TF-IDF Palavras)", "Otimização": "GridSearchCV (ngram, C)", "Macro F1": f1_logreg, "Acurácia": f"{acc_logreg*100:.2f}%"},
    {"Modelo": "3. Complement Naive Bayes", "Otimização": "GridSearchCV (alpha, norm)", "Macro F1": f1_cnb, "Acurácia": f"{acc_cnb*100:.2f}%"},
    {"Modelo": "4. LinearSVC (TF-IDF Palavras)", "Otimização": "GridSearchCV (ngram, C)", "Macro F1": f1_svc, "Acurácia": f"{acc_svc*100:.2f}%"},
    {"Modelo": "5. LinearSVC (TF-IDF Caracteres)", "Otimização": "GridSearchCV (ngram, min_df)", "Macro F1": f1_char, "Acurácia": f"{acc_char*100:.2f}%"},
    {"Modelo": "6. HistGradientBoosting (Features Estruturais)", "Otimização": "GridSearchCV (lr, leaf_nodes)", "Macro F1": f1_hgb, "Acurácia": f"{acc_hgb*100:.2f}%"},
    {"Modelo": "7. Regressão Ordinal Ridge", "Otimização": "GridSearchCV (alpha) + Cortes", "Macro F1": f1_ridge, "Acurácia": f"{acc_ridge*100:.2f}%"},
    {"Modelo": "8. Embeddings Densos (MiniLM)", "Otimização": "GridSearchCV (C)", "Macro F1": f1_emb, "Acurácia": f"{acc_emb*100:.2f}%"},
    {"Modelo": "9. BERTimbau Transformer", "Otimização": "Busca Offline (lr, trunc)", "Macro F1": f1_bert, "Acurácia": f"{acc_bert*100:.2f}%"},
    {"Modelo": "10. Híbrido (LinearSVC + Precedentes k-NN)", "Otimização": "Grid Search (k, peso)", "Macro F1": f1_hibrido, "Acurácia": f"{acc_hibrido*100:.2f}%"}
])

print(tabela.to_string(index=False))

                                        Modelo                    Otimização  Macro F1 Acurácia
                       1. Baseline Majoritário             Nenhum (Baseline)  0.173990   35.32%
      2. Regressão Logística (TF-IDF Palavras)       GridSearchCV (ngram, C)  0.468951   47.12%
                     3. Complement Naive Bayes    GridSearchCV (alpha, norm)  0.467518   47.18%
                4. LinearSVC (TF-IDF Palavras)       GridSearchCV (ngram, C)  0.464018   46.57%
              5. LinearSVC (TF-IDF Caracteres)  GridSearchCV (ngram, min_df)  0.456291   46.02%
6. HistGradientBoosting (Features Estruturais) GridSearchCV (lr, leaf_nodes)  0.391043   40.04%
                    7. Regressão Ordinal Ridge GridSearchCV (alpha) + Cortes  0.442071   43.97%
                 8. Embeddings Densos (MiniLM)              GridSearchCV (C)  0.414421   41.95%
                      9. BERTimbau Transformer     Busca Offline (lr, trunc)  0.448321   44.97%
    10. Híbrido (LinearSVC + Precedentes